<a href="https://colab.research.google.com/github/Shivxnshjasathi/-privacy-policy/blob/main/Scholarship-model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import pandas as pd

# Load the dataset
file_path = '/content/dataset_combined.xlsx'
df = pd.read_excel(file_path)

# Display the first few rows and column information
print("Dataset Columns:", df.columns.tolist())
display(df.head())

Dataset Columns: ['Name', 'Education Qualification', 'Gender', 'Community', 'Religion', 'Exservice-men', 'Disability', 'Sports', 'Annual-Percentage', 'Income', 'India', 'Outcome']


,Name,Education Qualification,Gender,Community,Religion,Exservice-men,Disability,Sports,Annual-Percentage,Income,India,Outcome
0,INSPIRE Scholarship 2022-23 ? Scholarship for ...,Undergraduate,Male,General,Hindu,Yes,Yes,Yes,90-100,Upto 1.5L,In,1
1,INSPIRE Scholarship 2022-23 ? Scholarship for ...,Undergraduate,Male,General,Hindu,Yes,Yes,No,90-100,Upto 1.5L,In,1
2,INSPIRE Scholarship 2022-23 ? Scholarship for ...,Undergraduate,Male,General,Muslim,Yes,Yes,Yes,90-100,Upto 1.5L,In,1
3,INSPIRE Scholarship 2022-23 ? Scholarship for ...,Undergraduate,Male,General,Muslim,Yes,Yes,No,90-100,Upto 1.5L,In,1
4,INSPIRE Scholarship 2022-23 ? Scholarship for ...,Undergraduate,Male,General,Chirstian,Yes,Yes,Yes,90-100,Upto 1.5L,In,1


In [47]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Prepare Features (X) and Target (y)
# We'll use relevant columns for prediction
features = ['Education Qualification', 'Gender', 'Community', 'Religion', 'Annual-Percentage', 'Income']
target = 'Outcome'

# Drop rows with missing values in these specific columns
ml_df = df[features + [target]].dropna()

# 2. Encode Categorical Data
encoders = {}
for col in features:
    le = LabelEncoder()
    ml_df[col] = le.fit_transform(ml_df[col].astype(str))
    encoders[col] = le

X = ml_df[features]
y = ml_df[target]

# 3. Train the Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = DecisionTreeClassifier(max_depth=10, random_state=42)
model.fit(X_train, y_train)

# 4. Evaluate
y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.2%}")

Model Accuracy: 86.69%


In [48]:
def get_all_applicable_scholarships(user_profile):
    """
    Searches the entire dataset for scholarships matching user criteria with flexible filtering.
    """
    search_df = df.copy()

    # Convert columns to string and handle NaN for safe searching
    for col in ['Community', 'Gender', 'Education Qualification']:
        search_df[col] = search_df[col].astype(str).fillna('')

    # 1. Flexible Community Filter
    # Matches specific community, 'All', 'OC' (Open Category), or 'General'
    comm_query = str(user_profile['Community'])
    community_mask = (
        search_df['Community'].str.contains(comm_query, case=False) |
        search_df['Community'].str.contains('All', case=False) |
        search_df['Community'].str.contains('General', case=False) |
        search_df['Community'].str.contains('OC', case=False)
    )

    # 2. Flexible Gender Filter
    gender_query = str(user_profile['Gender'])
    gender_mask = (
        search_df['Gender'].str.contains(gender_query, case=False) |
        search_df['Gender'].str.contains('All', case=False)
    )

    # 3. Education Filter
    edu_query = str(user_profile['Education Qualification'])
    edu_mask = search_df['Education Qualification'].str.contains(edu_query, case=False)

    # Apply filters
    final_matches = search_df[community_mask & gender_mask & edu_mask]

    # If no results found with education filter, broaden search to just Community and Gender
    if final_matches.empty:
        final_matches = search_df[community_mask & gender_mask]

    # 4. Use ML to rank results
    if not final_matches.empty:
        ranking_data = final_matches[features].copy()
        for col in features:
            ranking_data[col] = encoders[col].transform(ranking_data[col].astype(str))

        # Add a score based on AI prediction
        probs = model.predict_proba(ranking_data)[:, 1]
        final_matches['AI_Confidence'] = (probs * 100).round(2)
        final_matches = final_matches.sort_values(by='AI_Confidence', ascending=False)

    return final_matches

In [49]:
#@title 🎓 Optimized Scholarship Finder { run: "auto" }
edu = "PG" #@param ["10th", "12th", "UG", "PG"]
gen = "Female" #@param ["Male", "Female", "All"]
comm = "General" #@param ["General", "OBC", "SC/ST", "Minority", "All"]
marks_input = "80-90" #@param ["60-70", "70-80", "80-90", "90-100"]

profile = {
    'Education Qualification': edu,
    'Gender': gen,
    'Community': comm,
    'Religion': 'All',
    'Annual-Percentage': marks_input,
    'Income': 'Below 2L'
}

all_matches = get_all_applicable_scholarships(profile)

print(f"--- Recommendations for {edu} student ({comm}) ---")
if not all_matches.empty:
    print(f"Found {len(all_matches)} scholarships you might be eligible for.")
    # Remove duplicates if names are repetitive in dataset
    display_df = all_matches.drop_duplicates(subset=['Name']).head(15)
    display(display_df[['Name', 'Education Qualification', 'Community', 'Gender', 'AI_Confidence']])
else:
    print("Still no matches found. Please check if the dataset has 'All' categories.")

--- Recommendations for PG student (General) ---
Found 30720 scholarships you might be eligible for.


/tmp/ipykernel_841/1308345043.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_matches['AI_Confidence'] = (probs * 100).round(2)


,Name,Education Qualification,Community,Gender,AI_Confidence
92368,Glow and lovely Career Foundation Scholarship,Postgraduate,General,Female,24.32
167260,Pragati Scholarship ? AICTE-Scholarship Scheme...,Postgraduate,General,Female,24.32
19792,INSPIRE Scholarship 2022-23 ? Scholarship for ...,Postgraduate,General,Female,24.32
216412,Indira Gandhi Scholarship for Single Girl Chil...,Postgraduate,General,Female,24.32
44368,Abdul Kalam Technology Innovation National Fel...,Postgraduate,General,Female,24.32
67792,AAI Sports Scholarship Scheme in India 2022-23,Postgraduate,General,Female,24.32
240592,National Overseas Scholarship Scheme 2021-22,Postgraduate,General,Female,24.32
142676,ONGC Sports Scholarship Scheme 2022-23,Postgraduate,General,Female,24.32
116944,National Fellowship for Persons with Disabilities,Postgraduate,General,Female,24.32
191064,Dr. Ambedkar post matric Scholarship,Postgraduate,General,Female,24.32
